In [64]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt



# Load data


In [65]:
# Load data
df = pd.read_csv("sample_data/global_climate_energy_2020_2024.csv")


In [66]:
TARGET_COL = "energy_consumption"
df.head()

,date,country,avg_temperature,humidity,co2_emission,energy_consumption,renewable_share,urban_population,industrial_activity_index,energy_price
0,2020-01-01,Germany,28.29,31.08,212.63,11348.75,14.42,76.39,51.22,83.93
1,2020-01-02,Germany,28.38,37.94,606.05,4166.64,5.63,86.26,78.27,110.40
2,2020-01-03,Germany,28.74,57.67,268.72,4503.80,14.20,75.92,48.96,173.58
3,2020-01-04,Germany,26.66,51.34,167.32,3259.13,13.84,63.15,97.42,89.13
4,2020-01-05,Germany,26.81,65.38,393.89,7023.72,6.93,76.02,81.89,40.60


In [67]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [68]:
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
print(df.describe())


Missing values:
date                         0
country                      0
avg_temperature              0
humidity                     0
co2_emission                 0
energy_consumption           0
renewable_share              0
urban_population             0
industrial_activity_index    0
energy_price                 0
dtype: int64

Basic statistics:
       avg_temperature      humidity  co2_emission  energy_consumption  \
count     36540.000000  36540.000000  36540.000000        36540.000000   
mean         13.580868     59.971469    445.820452         7295.904857   
std          10.077249     17.303103    234.360906         3693.928504   
min          -9.600000     30.000000     50.150000         1001.890000   
25%           5.630000     45.010000    248.675000         4184.177500   
50%          13.790000     59.990000    422.655000         6921.620000   
75%          20.840000     74.970000    628.422500        10175.110000   
max          38.710000     90.000000    999.85000

In [69]:
print("\n" + "="*60)
print("PREPROCESSING")
print("="*60)

# Drop rows with missing values
df = df.dropna()
print(f"Dataset shape after removing NaN: {df.shape}")

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from numerical columns
if TARGET_COL in numerical_cols:
    numerical_cols.remove(TARGET_COL)

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# For this dataset: date is date, country is categorical
# Keep date info but encode country
print(f"\nBefore encoding:")
print(f"  - date: {df['date'].dtype}")
print(f"  - country: unique values = {df['country'].nunique()}")



PREPROCESSING
Dataset shape after removing NaN: (36540, 10)

Categorical columns: ['date', 'country']
Numerical columns: ['avg_temperature', 'humidity', 'co2_emission', 'renewable_share', 'urban_population', 'industrial_activity_index', 'energy_price']

Before encoding:
  - date: object
  - country: unique values = 20


In [70]:
# Convert date to datetime and extract features
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter

print("\nExtracted date features:")
print(f"  - year, month, day, dayofweek, quarter")

# Drop the original date column (no longer needed)
df = df.drop(columns=['date'])

# One-hot encode 'country' (categorical variable)
df_encoded = pd.get_dummies(df, columns=['country'], drop_first=True)

print(f"\nAfter one-hot encoding country:")
print(f"  - New shape: {df_encoded.shape}")
print(f"  - New columns: {df_encoded.columns.tolist()}")


Extracted date features:
  - year, month, day, dayofweek, quarter

After one-hot encoding country:
  - New shape: (36540, 32)
  - New columns: ['avg_temperature', 'humidity', 'co2_emission', 'energy_consumption', 'renewable_share', 'urban_population', 'industrial_activity_index', 'energy_price', 'year', 'month', 'day', 'dayofweek', 'quarter', 'country_Brazil', 'country_Canada', 'country_China', 'country_France', 'country_Germany', 'country_India', 'country_Indonesia', 'country_Italy', 'country_Japan', 'country_Mexico', 'country_Netherlands', 'country_Norway', 'country_Poland', 'country_South Africa', 'country_Spain', 'country_Sweden', 'country_Turkey', 'country_United Kingdom', 'country_United States']


# Standarization

In [71]:
# ============================================
# CELL 4: STANDARDIZATION
# ============================================

print("\n" + "="*60)
print("STANDARDIZATION")
print("="*60)

# Separate features and target
X = df_encoded.drop(columns=[TARGET_COL])
y = df_encoded[[TARGET_COL]].copy()

print(f"Features shape before scaling: {X.shape}")
print(f"Target shape: {y.shape}")

# Standardize features
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Standardize target
target_scaler = StandardScaler()
y_scaled = target_scaler.fit_transform(y)
y_scaled = pd.Series(y_scaled.flatten(), name=TARGET_COL)

print("\nFeatures standardized (mean ≈ 0, std ≈ 1)")
print("\nFeature statistics after standardization:")
print(X_scaled.describe())
print("\nTarget statistics after standardization:")
print(y_scaled.describe())



STANDARDIZATION
Features shape before scaling: (36540, 31)
Target shape: (36540, 1)

Features standardized (mean ≈ 0, std ≈ 1)

Feature statistics after standardization:
       avg_temperature      humidity  co2_emission  renewable_share  \
count     3.654000e+04  3.654000e+04  3.654000e+04     3.654000e+04   
mean      2.800168e-17 -2.866284e-16  1.687879e-16     3.437985e-16   
std       1.000014e+00  1.000014e+00  1.000014e+00     1.000014e+00   
min      -2.300349e+00 -1.732168e+00 -1.688319e+00    -2.051477e+00   
25%      -7.890027e-01 -8.646816e-01 -8.412160e-01    -7.355722e-01   
50%       2.075322e-02  1.070990e-03 -9.884656e-02    -4.294118e-02   
75%       7.203585e-01  8.668235e-01  7.791596e-01     7.302937e-01   
max       2.493684e+00  1.735466e+00  2.364034e+00     2.797877e+00   

       urban_population  industrial_activity_index  energy_price  \
count      3.654000e+04               3.654000e+04  3.654000e+04   
mean      -1.071648e-15               5.355322e-16 -1

In [72]:
print("data set after preprocessing:")
print(X_scaled.head())
print(y_scaled.head())

data set after preprocessing:
   avg_temperature  humidity  co2_emission  renewable_share  urban_population  \
0         1.459658 -1.669751     -0.995019        -0.285690          0.162845   
1         1.468589 -1.273284      0.683697        -1.933383          1.304509   
2         1.504313 -0.133011     -0.755684        -0.326929          0.108480   
3         1.297905 -0.498846     -1.188356        -0.394412         -1.368626   
4         1.312790  0.312580     -0.221586        -1.689697          0.120047   

   industrial_activity_index  energy_price     year     month       day  ...  \
0                  -1.093369     -0.637481 -1.41344 -1.600676 -1.673987  ...   
1                   0.467096     -0.099229 -1.41344 -1.600676 -1.560381  ...   
2                  -1.223744      1.185500 -1.41344 -1.600676 -1.446776  ...   
3                   1.571824     -0.531742 -1.41344 -1.600676 -1.333170  ...   
4                   0.675927     -1.518572 -1.41344 -1.600676 -1.219565  ...   

  

In [73]:
#split data into train and test sets and validation set
print("\n" + "="*60)
print("DATA SPLITTING")
print("="*60)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")


DATA SPLITTING
Training set: 21924 samples
Validation set: 7308 samples
Test set: 7308 samples
Number of features: 31


In [74]:
print("\n" + "="*60)
print("CONVERTING TO PYTORCH TENSORS")
print("="*60)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print(f"X_train tensor shape: {X_train_tensor.shape}")
print(f"y_train tensor shape: {y_train_tensor.shape}")
print(f"X_val tensor shape: {X_val_tensor.shape}")
print(f"X_test tensor shape: {X_test_tensor.shape}")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)
print(f"Ready for model training with {X_train.shape[1]} features")


CONVERTING TO PYTORCH TENSORS
X_train tensor shape: torch.Size([21924, 31])
y_train tensor shape: torch.Size([21924, 1])
X_val tensor shape: torch.Size([7308, 31])
X_test tensor shape: torch.Size([7308, 31])

PREPROCESSING COMPLETE!
Ready for model training with 31 features


In [75]:
# Store original min/max values for denormalization later
y_min = y_train_tensor.min().item()
y_max = y_train_tensor.max().item()
y_range = y_max - y_min

print(f"\nOriginal y_train range:")
print(f"  Min: {y_min:.4f}")
print(f"  Max: {y_max:.4f}")
print(f"  Range: {y_range:.4f}")

# Normalize all targets to [0, 1] using Min-Max scaling
y_train_norm = (y_train_tensor - y_min) / y_range
y_val_norm = (y_val_tensor - y_min) / y_range
y_test_norm = (y_test_tensor - y_min) / y_range

print(f"\nAfter normalization to [0, 1]:")
print(f"  y_train: min={y_train_norm.min():.4f}, max={y_train_norm.max():.4f}, mean={y_train_norm.mean():.4f}")
print(f"  y_val:   min={y_val_norm.min():.4f}, max={y_val_norm.max():.4f}, mean={y_val_norm.mean():.4f}")
print(f"  y_test:  min={y_test_norm.min():.4f}, max={y_test_norm.max():.4f}, mean={y_test_norm.mean():.4f}")

print("\n" + "="*80)


Original y_train range:
  Min: -1.7039
  Max: 2.3557
  Range: 4.0596

After normalization to [0, 1]:
  y_train: min=0.0000, max=1.0000, mean=0.4183
  y_val:   min=0.0006, max=0.9999, mean=0.4230
  y_test:  min=-0.0000, max=1.0000, mean=0.4207



In [82]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import copy

# ============================
# STRONGER MODEL
# ============================

class ImprovedEnergyModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.4):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim//2),
            nn.LayerNorm(hidden_dim//2),
            nn.GELU(),
            nn.Dropout(dropout/2),

            nn.Linear(hidden_dim//2, 1)
        )

    def forward(self, x):
        return self.net(x)

# ============================
# DATA LOADERS + TRAINING
# ============================

batch_size = 1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

model = ImprovedEnergyModel(input_dim=X_train.shape[1], hidden_dim=512, dropout=0.35).to(device)

criterion = nn.HuberLoss(delta=0.8)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.4, patience=12)

# Early stopping
best_val_loss = float('inf')
best_state = None
patience = 50
counter = 0

train_losses, val_losses = [], []

print("Training improved model...")

for epoch in range(400):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb), yb).item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if epoch % 20 == 0 or counter == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.5f} | Val: {val_loss:.5f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
print(f"\nBest Validation Loss: {best_val_loss:.5f}")

# ============================
# FINAL EVALUATION
# ============================

model.eval()
preds, trues = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb)
        preds.extend(pred.cpu().numpy().flatten())
        trues.extend(yb.numpy().flatten())

y_true_orig = target_scaler.inverse_transform(np.array(trues).reshape(-1, 1)).flatten()
y_pred_orig = target_scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
mae = mean_absolute_error(y_true_orig, y_pred_orig)
r2 = r2_score(y_true_orig, y_pred_orig)

print("\n" + "="*70)
print("FINAL RESULTS")
print(f"Test RMSE : {rmse:,.2f}")
print(f"Test MAE  : {mae:,.2f}")
print(f"Test R²   : {r2:.4f}")

Training improved model...
Epoch   1 | Train: 0.41918 | Val: 0.41021 | LR: 3.00e-04
Epoch   2 | Train: 0.40041 | Val: 0.40301 | LR: 3.00e-04
Epoch   3 | Train: 0.39571 | Val: 0.40231 | LR: 3.00e-04
Epoch   4 | Train: 0.39353 | Val: 0.39962 | LR: 3.00e-04
Epoch   5 | Train: 0.39205 | Val: 0.39725 | LR: 3.00e-04
Epoch   6 | Train: 0.38932 | Val: 0.39527 | LR: 3.00e-04
Epoch   7 | Train: 0.38671 | Val: 0.39223 | LR: 3.00e-04
Epoch   8 | Train: 0.38544 | Val: 0.38892 | LR: 3.00e-04
Epoch   9 | Train: 0.38367 | Val: 0.38557 | LR: 3.00e-04
Epoch  10 | Train: 0.38031 | Val: 0.38237 | LR: 3.00e-04
Epoch  11 | Train: 0.37779 | Val: 0.37953 | LR: 3.00e-04
Epoch  12 | Train: 0.37525 | Val: 0.37716 | LR: 3.00e-04
Epoch  14 | Train: 0.37180 | Val: 0.37411 | LR: 3.00e-04
Epoch  15 | Train: 0.37221 | Val: 0.37371 | LR: 3.00e-04
Epoch  16 | Train: 0.36802 | Val: 0.37233 | LR: 3.00e-04
Epoch  19 | Train: 0.36709 | Val: 0.37191 | LR: 3.00e-04
Epoch  20 | Train: 0.36570 | Val: 0.36985 | LR: 3.00e-04
Epoc

In [83]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

xgb = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=9,
    subsample=0.85,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    random_state=42,
    tree_method='hist'
)

xgb.fit(X_train, y_train)

pred_scaled = xgb.predict(X_test)
pred_orig = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
true_orig = target_scaler.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

print("XGBoost R²:", r2_score(true_orig, pred_orig))

XGBoost R²: 0.08335734081385637
